> **TÍTULO:** CyberSentinel-IoT: Sistema Integrado de Detecção de Anomalias em Dispositivos IoT com Aprendizado de Máquina e Sumarização por GenAI
>
> **DATASET:** [Bot-IoT](http://research.unsw.edu.au/projects/bot-iot-dataset) (Training-set / Testing-set - 10 best features)


In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

np.random.seed(42)

In [3]:
df_train = pd.read_csv("../data/Bot-IoT/UNSW_2018_IoT_Botnet_Final_10_best_Training.csv")
df_test  = pd.read_csv("../data/Bot-IoT/UNSW_2018_IoT_Botnet_Final_10_best_Testing.csv")

# Marcamos a origem de cada registro antes de combinar, para podermos separar novamente se necessário
df_train["origem"] = "train"
df_test["origem"]  = "test"

df_full = pd.concat([df_train, df_test], axis=0, ignore_index=True)

print(f"Treino : {df_train.shape[0]:,} linhas x {df_train.shape[1]} colunas".replace(",", "."))
print(f"Teste  : {df_test.shape[0]:,} linhas x {df_test.shape[1]} colunas".replace(",", "."))
print(f"Total  : {df_full.shape[0]:,} linhas x {df_full.shape[1]} colunas".replace(",", "."))

Treino : 2.934.817 linhas x 20 colunas
Teste  : 733.705 linhas x 20 colunas
Total  : 3.668.522 linhas x 20 colunas


In [4]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3668522 entries, 0 to 3668521
Data columns (total 20 columns):
 #   Column             Dtype  
---  ------             -----  
 0   pkSeqID            int64  
 1   proto              object 
 2   saddr              object 
 3   sport              object 
 4   daddr              object 
 5   dport              object 
 6   seq                int64  
 7   stddev             float64
 8   N_IN_Conn_P_SrcIP  int64  
 9   min                float64
 10  state_number       int64  
 11  mean               float64
 12  N_IN_Conn_P_DstIP  int64  
 13  drate              float64
 14  srate              float64
 15  max                float64
 16  attack             int64  
 17  category           object 
 18  subcategory        object 
 19  origem             object 
dtypes: float64(6), int64(6), object(8)
memory usage: 559.8+ MB


### _Features_ do dataset

| Nome da Feature | Tipo | Descrição |
| :--- | :--- | :--- |
| **pkSeqID** | `int64` | Identificador único da linha / registro |
| **proto** | `object` | Representação em texto do protocolo de transação no fluxo de rede |
| **saddr** | `object` | Endereço IP de origem (*Source IP*) |
| **sport** | `object` | Número da porta de origem (*Source port*) |
| **daddr** | `object` | Endereço IP de destino (*Destination IP*) |
| **dport** | `object` | Número da porta de destino (*Destination port*) |
| **seq** | `int64` | Número de sequência do Argus |
| **stddev** | `float64` | Desvio padrão da duração dos registros agregados |
| **N_IN_Conn_P_SrcIP** | `int64` | Número de conexões de entrada por IP de origem |
| **min** | `float64` | Duração mínima dos registros agregados |
| **state_number** | `int64` | Representação numérica do estado da transação (*state*) |
| **mean** | `float64` | Duração média dos registros agregados |
| **N_IN_Conn_P_DstIP** | `int64` | Número de conexões de entrada por IP de destino |
| **drate** | `float64` | Pacotes por segundo no sentido destino-para-origem |
| **srate** | `float64` | Pacotes por segundo no sentido origem-para-destino |
| **max** | `float64` | Duração máxima dos registros agregados |
| **attack** | `int64` | Rótulo da classe: `0` para tráfego Normal, `1` para Tráfego de Ataque |
| **category** | `object` | Categoria do tráfego de rede |
| **subcategory** | `object` | Subcategoria do tráfego de rede |
| **origem** | `object` | Metadado/Identificador da fonte/origem dos dados do registro |

In [5]:
# Separando os campos categóricos (`object`) e numéricos automaticamente, para conferência.
colunas_categoricas = df_full.select_dtypes(include="object").columns.tolist()
colunas_numericas   = df_full.select_dtypes(include=[np.number]).columns.tolist()

print(f"Campos categóricos ({len(colunas_categoricas)}):\n{colunas_categoricas}\n")
print(f"Campos numéricos ({len(colunas_numericas)}):\n{colunas_numericas}")

Campos categóricos (8):
['proto', 'saddr', 'sport', 'daddr', 'dport', 'category', 'subcategory', 'origem']

Campos numéricos (12):
['pkSeqID', 'seq', 'stddev', 'N_IN_Conn_P_SrcIP', 'min', 'state_number', 'mean', 'N_IN_Conn_P_DstIP', 'drate', 'srate', 'max', 'attack']


In [6]:
# df.sample() mostram exemplos reais de linhas, permitindo inspecionar visualmente o formato dos dados
df_full.sample(5, random_state=42)

,pkSeqID,proto,saddr,sport,daddr,dport,seq,stddev,N_IN_Conn_P_SrcIP,min,state_number,mean,N_IN_Conn_P_DstIP,drate,srate,max,attack,category,subcategory,origem
995784,3637643,udp,192.168.100.148,63063,192.168.100.5,28840,13988,0.000000,18,0.000000,4,0.000000,86,0.000000,0.000000,0.000000,1,Reconnaissance,Service_Scan,train
170580,613899,tcp,192.168.100.148,5071,192.168.100.6,80,88118,0.022247,23,0.000000,1,0.022042,23,0.017667,0.053008,0.048346,1,DoS,TCP,train
3141497,3633479,tcp,192.168.100.149,51456,192.168.100.5,8800,9790,0.000000,100,0.000918,1,0.000918,100,0.000000,0.000000,0.000918,1,Reconnaissance,Service_Scan,test
2925715,3667897,tcp,192.168.100.149,2166,192.168.100.5,1,8658,0.000000,25,0.000156,1,0.000156,100,0.000000,0.000000,0.000156,1,Reconnaissance,Service_Scan,train
3614039,2654500,udp,192.168.100.148,6724,192.168.100.3,80,25883,0.314900,100,3.481778,4,3.926956,100,0.000000,0.532142,4.159831,1,DDoS,UDP,test


In [7]:
# Amostra de valores únicos para cada campo categórico relevante:
for col in ["proto", "category", "subcategory"]:
    valores = df_full[col].unique()
    print(f"{col} ({df_full[col].nunique()} valores únicos): {sorted(map(str, valores))[:15]}"
          f"{' ...' if df_full[col].nunique() > 15 else ''}")

proto (5 valores únicos): ['arp', 'icmp', 'ipv6-icmp', 'tcp', 'udp']
category (5 valores únicos): ['DDoS', 'DoS', 'Normal', 'Reconnaissance', 'Theft']
subcategory (8 valores únicos): ['Data_Exfiltration', 'HTTP', 'Keylogging', 'Normal', 'OS_Fingerprint', 'Service_Scan', 'TCP', 'UDP']


In [8]:
resumo_numerico = df_full[colunas_numericas].describe().T
resumo_numerico["range"] = resumo_numerico["max"] - resumo_numerico["min"]
resumo_numerico[["min", "max", "mean", "std", "range"]].sort_values("range", ascending=False).head(15)

,min,max,mean,std,range
pkSeqID,1.0,3.668522e+06,1.834262e+06,1.059011e+06,3.668521e+06
srate,0.0,1.000000e+06,2.955111e+00,7.245425e+02,1.000000e+06
seq,1.0,2.622120e+05,1.213204e+05,7.579428e+04,2.622110e+05
drate,0.0,5.882353e+04,4.455046e-01,6.028863e+01,5.882353e+04
N_IN_Conn_P_DstIP,1.0,1.000000e+02,9.245168e+01,1.817643e+01,9.900000e+01
N_IN_Conn_P_SrcIP,1.0,1.000000e+02,8.253848e+01,2.439739e+01,9.900000e+01
state_number,1.0,1.100000e+01,3.134390e+00,1.186971e+00,1.000000e+01
max,0.0,4.999999e+00,3.020015e+00,1.860877e+00,4.999999e+00
mean,0.0,4.981882e+00,2.231063e+00,1.517728e+00,4.981882e+00
min,0.0,4.980471e+00,1.017540e+00,1.483688e+00,4.980471e+00


In [9]:
# Verificando balanceamento das classes (attack, category e subcategory)
print("Distribuição de 'attack' (0 = Normal, 1 = Ataque):")
print(df_full["attack"].value_counts())
print(f"Proporção de ataques: {df_full['attack'].mean():.2%}\n")

print("Distribuição de 'category':")
print(df_full["category"].value_counts())

print("\nDistribuição de 'subcategory':")
print(df_full["subcategory"].value_counts())

Distribuição de 'attack' (0 = Normal, 1 = Ataque):
attack
1    3668045
0        477
Name: count, dtype: int64
Proporção de ataques: 99.99%

Distribuição de 'category':
category
DDoS              1926624
DoS               1650260
Reconnaissance      91082
Normal                477
Theft                  79
Name: count, dtype: int64

Distribuição de 'subcategory':
subcategory
UDP                  1981230
TCP                  1593180
Service_Scan           73168
OS_Fingerprint         17914
HTTP                    2474
Normal                   477
Keylogging                73
Data_Exfiltration          6
Name: count, dtype: int64


In [10]:
# Removendo linhas duplicadas
qtd_duplicadas = df_full.duplicated().sum()
print(f"Linhas duplicadas encontradas: {qtd_duplicadas}")

if qtd_duplicadas > 0:
    df_full = df_full.drop_duplicates().reset_index(drop=True)
    print(f"Duplicadas removidas. Novo shape: {df_full.shape}")
else:
    print("Nenhuma duplicata encontrada.")

Linhas duplicadas encontradas: 0
Nenhuma duplicata encontrada.


### Padronização de campos categóricos

Removemos espaços em branco acidentais e uniformizamos a caixa (lowercase), prática comum para evitar que `'TCP'` e `'tcp '` sejam tratados como categorias diferentes.


In [22]:
for col in ["proto", "category", "subcategory"]:
    antes = df_full[col].nunique()
    df_full[col] = df_full[col].astype(str).str.strip().str.lower()
    depois = df_full[col].nunique()
    print(f"{col}: {antes} -> {depois} categorias únicas após strip() e lower()")

proto: 5 -> 5 categorias únicas após strip() e lower()
category: 5 -> 5 categorias únicas após strip() e lower()
subcategory: 8 -> 8 categorias únicas após strip() e lower()


### Verificação de valores ausentes (nulos)


In [24]:
nulos_por_coluna = df_full.isnull().sum()
total_nulos = nulos_por_coluna.sum()

print(f"Total de valores nulos no dataset: {total_nulos}")
if total_nulos > 0:
    print(nulos_por_coluna[nulos_por_coluna > 0])
else:
    print("Nenhum valor nulo (NaN) real no dataset original.")

Total de valores nulos no dataset: 0
Nenhum valor nulo (NaN) real no dataset original.


### Filtro 1: Somente tráfego malicioso (`label == 1`)

Isola todos os registros classificados como ataque, independentemente da categoria: separa o que é "estranho" do que é tráfego normal.


In [25]:
df_ataques = df_full[df_full["attack"] == 1]

print(f"Registros de ataque: {len(df_ataques):,} de {len(df_full):,} ({len(df_ataques)/len(df_full):.2%})".replace(",", "."))
df_ataques.head(3)

Registros de ataque: 3.668.045 de 3.668.522 (99.99%)


,pkSeqID,proto,saddr,sport,daddr,dport,seq,stddev,N_IN_Conn_P_SrcIP,min,state_number,mean,N_IN_Conn_P_DstIP,drate,srate,max,attack,category,subcategory,origem
0,3142762,udp,192.168.100.150,6551,192.168.100.3,80,251984,1.900363,100,0.00000,4,2.687519,100,0.0,0.494549,4.031619,1,ddos,udp,train
1,2432264,tcp,192.168.100.150,5532,192.168.100.3,80,256724,0.078003,38,3.85693,3,3.934927,100,0.0,0.256493,4.012924,1,ddos,tcp,train
2,1976315,tcp,192.168.100.147,27165,192.168.100.3,80,62921,0.268666,100,2.97410,3,3.341429,100,0.0,0.294880,3.609205,1,ddos,tcp,train


### Filtro 2: Somente tráfego normal (`label == 0`)

Filtro usado como baseline de comparação (ex.: para treinar o modelo de detecção de anomalia apenas com tráfego legítimo, técnica comum em Isolation Forest/Autoencoders).


In [26]:
df_normal = df_full[df_full["attack"] == 0]

print(f"Registros normais: {len(df_normal):,} de {len(df_full):,} ({len(df_normal)/len(df_full):.2%})".replace(",", "."))

Registros normais: 477 de 3.668.522 (0.01%)


### Filtro 3: Uma família específica de ataque (`category == 'ddos'`)

Isola apenas os registros da categoria "ddos", permitindo estudar o comportamento característico desse tipo de ataque (ex.: alto volume de pacotes em curto intervalo de tempo).


In [27]:
df_ddos = df_full[df_full["category"].str.lower() == "ddos"]

print(f"Registros da categoria DDoS: {len(df_ddos):,}".replace(",", "."))
df_ddos.describe()


Registros da categoria DDoS: 1.926.624


,pkSeqID,seq,stddev,N_IN_Conn_P_SrcIP,min,state_number,mean,N_IN_Conn_P_DstIP,drate,srate,max,attack
count,1.926624e+06,1.926624e+06,1.926624e+06,1.926624e+06,1.926624e+06,1.926624e+06,1.926624e+06,1.926624e+06,1.926624e+06,1.926624e+06,1.926624e+06,1926624.0
mean,2.613572e+06,1.232739e+05,9.453554e-01,8.195921e+01,1.301874e+00,3.043381e+00,2.549634e+00,9.990743e+01,7.041794e-03,5.153571e-01,3.348972e+00,1.0
std,5.561686e+05,7.378819e+04,7.517980e-01,2.420960e+01,1.491054e+00,1.191124e+00,1.404212e+00,2.195074e+00,3.566600e-02,6.061590e-01,1.646568e+00,0.0
min,1.650261e+06,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.0
25%,2.131917e+06,6.009200e+04,1.230160e-01,6.800000e+01,0.000000e+00,3.000000e+00,2.016244e+00,1.000000e+02,0.000000e+00,2.498630e-01,3.262079e+00,1.0
50%,2.613572e+06,1.202990e+05,9.376220e-01,9.700000e+01,1.696500e-01,3.000000e+00,2.759816e+00,1.000000e+02,0.000000e+00,4.100830e-01,4.057637e+00,1.0
75%,3.095228e+06,1.831780e+05,1.690094e+00,1.000000e+02,2.763413e+00,4.000000e+00,3.659881e+00,1.000000e+02,0.000000e+00,6.346650e-01,4.172465e+00,1.0
max,3.576884e+06,2.621680e+05,2.496763e+00,1.000000e+02,4.980471e+00,7.000000e+00,4.981882e+00,1.000000e+02,1.064578e+00,7.339449e+00,4.999999e+00,1.0


### Filtro 4: Protocolo específico (`subcategory == 'tcp'`)

Filtra apenas conexões que usam o protocolo TCP.

In [28]:
df_tcp = df_full[df_full["subcategory"] == "tcp"]

print(f"Conexões TCP: {len(df_tcp):,} de {len(df_full):,} ({len(df_tcp)/len(df_full):.2%})".replace(",", "."))

Conexões TCP: 1.593.180 de 3.668.522 (43.43%)
